In [1]:
from websocket import create_connection
import json
import pandas as pd
import re
import datetime as dt
import textwrap
import time
import threading

In [2]:
import random
import string

def generate_session_id():
    return ''.join(random.choices(string.ascii_letters + string.digits, k=16))

def robust_websocket_connection():
    session_counter = 0
    
    while True:
        try:
            session_counter += 1
            session_id = f"qs_session_{generate_session_id()}"
            
            print(f"[{time.strftime('%H:%M:%S')}] Starting session #{session_counter}: {session_id}")
            
            socket = 'wss://widgetdata.tradingview.com/socket.io/websocket'
            ws = create_connection(socket)
            
            def create_msg(ws, fun, arg):
                ms = json.dumps({"m": fun,"p": arg})
                msg = '~m~' + str(len(ms)) + '~m~' + ms
                ws.send(msg)
            
            # Setup met unieke session ID
            create_msg(ws, 'quote_create_session', [session_id])
            create_msg(ws, 'quote_set_fields', [session_id, "lp", "chp", "ch"])
            create_msg(ws, 'quote_add_symbols', [session_id, "NASDAQ:AAPL"])
            
            connection_start = time.time()
            max_connection_time = 120  # Max 2 minuten per sessie
            
            while (time.time() - connection_start) < max_connection_time:
                res = ws.recv()
                
                if '~h~' in res:
                    ws.send(res)
                    continue
                
                if 'critical_error' in res:
                    print("Critical error detected, switching session...")
                    break
                
                if 'lp' in res:
                    price_match = re.search(r'"lp":([\d.]+)', res)
                    if price_match:
                        price = float(price_match.group(1))
                        current_time = time.strftime('%H:%M:%S')
                        print(f"[{current_time}] Session #{session_counter} - AAPL: ${price}")
            
            print(f"Session #{session_counter} completed, switching to new session...")
            ws.close()
            time.sleep(2)  # Korte pauze tussen sessies
            
        except KeyboardInterrupt:
            print("\n🛑 Stopped by user")
            break
        except Exception as e:
            print(f"Session #{session_counter} error: {e}, restarting...")
            time.sleep(3)

# Start roterend systeem
robust_websocket_connection()

[08:39:21] Starting session #1: qs_session_WonkZQ54sdmi3Oed
[08:39:21] Session #1 - AAPL: $262.82

🛑 Stopped by user


In [4]:
#verbinding maken met websocket als de markt open gaat
#verbinding verbreken wanneer markt dicht gaat
#als er socket connection error komt verbinding terug opstarten

socket = 'wss://widgetdata.tradingview.com/socket.io/websocket'

#quote data
msg3 = '~m~83~m~{"m":"quote_create_session","p":["qs_snapshoter_basic-symbol-quotes_oIZmIDmqmRI0"]}'
msg4 = '~m~448~m~{"m":"quote_set_fields","p":["qs_snapshoter_basic-symbol-quotes_oIZmIDmqmRI0","pro_name","base_name","short_name","description","type","exchange","typespecs","listed_exchange","lp","country_code","provider_id","symbol-primaryname","logoid","base-currency-logoid","currency-logoid","source-logoid","update_mode","pro_perm","source","source2","pricescale","minmov","fractional","visible-plots-set","local_description","language","underlying-symbol"]}'	
msg5 = '~m~94~m~{"m":"quote_add_symbols","p":["qs_snapshoter_basic-symbol-quotes_oIZmIDmqmRI0","NASDAQ:AAPL"]}'

messages = []
ws = create_connection(socket)

def create_msg(ws, fun, arg):
    ms = json.dumps({"m": fun,"p": arg})
    msg = '~m~' + str(len(ms)) + '~m~' + ms
    ws.send(msg)

create_msg(ws, 'quote_create_session', ["qs_snapshoter_basic-symbol-quotes_oIZmIDmqmRI0"])
#initialiseert de websocket verbinding en geeft alle info die nodig is om data op te vragen
create_msg(ws, 'quote_set_fields', ["qs_snapshoter_basic-symbol-quotes_oIZmIDmqmRI0","pro_name","base_name","short_name","description","type","exchange","typespecs","listed_exchange","lp","country_code","provider_id","symbol-primaryname","logoid","base-currency-logoid","currency-logoid","source-logoid","update_mode","pro_perm","source","source2","pricescale","minmov","fractional","visible-plots-set","local_description","language","underlying-symbol"])
create_msg(ws, 'quote_add_symbols', ["qs_snapshoter_basic-symbol-quotes_oIZmIDmqmRI0","NASDAQ:AAPL"])


#def listen_to_websocket():
while True:
    res = ws.recv()
    messages.append(res)
    print(res)

    if '~h~' in res:
        print('server ping received:', res)
        ws.send(res)
        print('pong sent:', res)

~m~293~m~{"session_id":"0.27869.1171_lon1-charts-wgt-1-tvbs-yw5ld-3","timestamp":1760112180,"timestampMs":1760112180168,"release":"release_208-76","studies_metadata_hash":"44aae310c18bc00c093e2de428dbee1bd4796e33","auth_scheme_vsn":2,"protocol":"json","via":"93.123.102.189:443","javastudies":["3.66"]}
~m~787~m~{"m":"qsd","p":["qs_snapshoter_basic-symbol-quotes_oIZmIDmqmRI0",{"n":"NASDAQ:AAPL","s":"ok","v":{"visible-plots-set":"ohlcv","update_mode":"streaming","typespecs":["common"],"type":"stock","symbol-primaryname":"NASDAQ:AAPL","source2":{"country":"US","description":"Cboe One","exchange-type":"exchange","id":"BATS","name":"Cboe One","url":"https://markets.cboe.com/us/equities/overview/"},"source-logoid":"source/NASDAQ","short_name":"AAPL","provider_id":"ice","pro_perm":"nasdaq","pro_name":"NASDAQ:AAPL","pricescale":100,"minmov":1,"lp":248.14,"logoid":"apple","local_description":"Apple Inc.","listed_exchange":"NASDAQ","language":"en","fractional":false,"exchange":"Cboe One","descrip

KeyboardInterrupt: 